In [ ]:
import json
import pandas as pd
from pathlib import Path

# ============================================================
# PARCO AUTOVETTURE PER PROVINCIA E ALIMENTAZIONE — ANNO 2025
# ============================================================
# Fonte: ACI "Open Parco Veicoli" (opv.aci.it), non il dataset
# ACI/MIT del 2019 già usato in auto_da_ricaricare.ipynb.
#
# Perché un notebook separato: questa fonte è più aggiornata (2025
# contro il 2019 dell'altro dataset) ma ha una granularità diversa
# — provincia, non singolo veicolo — e riguarda solo la categoria
# AUTOVETTURE (non moto/altri veicoli come il dataset ACI/MIT).
# Va quindi trattata come fonte a parte, non semplicemente accodata
# all'altra.
#
# Come sono stati ottenuti questi dati: opv.aci.it è una dashboard
# Pentaho/CDF interattiva (non un file scaricabile direttamente via
# URL). I filtri usati sono stati: Anno=2025, Dimensioni=ALIMENTAZIONE
# + SOLO PROVINCE, Categorie=AUTOVETTURE. I dati della tabella già
# calcolata dal componente sono stati letti direttamente dal suo stato
# interno (rawData) via browser e salvati in
# parco_circolante_2025_ACI_OPV_raw.json, che carichiamo qui.

CARTELLA = Path(".")
RAW_PATH = CARTELLA / "parco_circolante_2025_ACI_OPV_raw.json"

with open(RAW_PATH) as f:
    raw = json.load(f)

colonne = [m["colName"].strip() for m in raw["metadata"]]
df_raw = pd.DataFrame(raw["resultset"], columns=colonne)
print(f"Righe grezze: {len(df_raw)}")
df_raw.head()

Righe grezze: 109


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,Anno,Provincia,AL,BE,BG,BM,EL,GA,GG,IB,IBG,IBM,IG,IGG,ME,ND,Totale
0,2025,TORINO,35,734.272,207.429,16.359,17.377,431.568,6,227.734,93,0,12.735,0,2.004,33,1.649.645
1,2025,VERCELLI,2,57.846,10.423,498,753,42.695,1,10.053,5,0,905,0,149,3,123.333
2,2025,NOVARA,6,125.242,19.663,1.654,2.302,78.558,0,24.094,9,0,2.262,0,597,4,254.391
3,2025,CUNEO,10,180.420,35.510,1.100,2.879,178.749,2,32.066,23,0,4.058,0,312,11,435.140
4,2025,ASTI,2,67.858,14.280,1.126,949,58.933,0,10.452,7,0,1.080,0,163,2,154.852


In [ ]:
# Pulizia:
# - i numeri arrivano come stringhe in formato italiano ("." separatore
#   delle migliaia, es. "734.272"), oppure già come interi quando il
#   valore originale nel resultset era 0 — normalizziamo tutto a int;
# - la riga "Totale" (indice Anno vuoto) è il totale Italia già
#   calcolato dalla dashboard: la teniamo da parte come riferimento
#   invece di lasciarla mischiata alle province;
# - la riga "NON DEFINITO" rappresenta veicoli senza provincia
#   associata nell'anagrafica ACI: la teniamo ma separata dalle vere
#   province, per non falsare eventuali join con dati geografici.

def to_int(x):
    if isinstance(x, str):
        x = x.strip().replace(".", "")
        return int(x) if x else 0
    return int(x)

colonne_numeriche = [c for c in colonne if c not in ("Anno", "Provincia")]

df = df_raw.copy()
for c in colonne_numeriche:
    df[c] = df[c].apply(to_int)

totale_italia = df[df["Provincia"] == "Totale"].iloc[0]
non_definito = df[df["Provincia"] == "NON DEFINITO"].iloc[0]
province_2025 = df[~df["Provincia"].isin(["Totale", "NON DEFINITO"])].copy()
province_2025["Anno"] = 2025

print(f"Province: {len(province_2025)}")
print(f"Totale AUTOVETTURE Italia 2025 (dalla dashboard): {totale_italia['Totale']:,}")
province_2025.head()

Province: 107
Totale AUTOVETTURE Italia 2025 (dalla dashboard): 41,795,962


,Anno,Provincia,AL,BE,BG,BM,EL,GA,GG,IB,IBG,IBM,IG,IGG,ME,ND,Totale
0,2025,TORINO,35,734272,207429,16359,17377,431568,6,227734,93,0,12735,0,2004,33,1649645
1,2025,VERCELLI,2,57846,10423,498,753,42695,1,10053,5,0,905,0,149,3,123333
2,2025,NOVARA,6,125242,19663,1654,2302,78558,0,24094,9,0,2262,0,597,4,254391
3,2025,CUNEO,10,180420,35510,1100,2879,178749,2,32066,23,0,4058,0,312,11,435140
4,2025,ASTI,2,67858,14280,1126,949,58933,0,10452,7,0,1080,0,163,2,154852


In [ ]:
# Colonna aggregata elettrico+ibrido per provincia.
# Sigle alimentazione (da legenda.html di opv.aci.it):
#   EL  = elettrico puro
#   IB  = ibrido benzina        IG  = ibrido gasolio       IM  = ibrido metano
#   IBG = ibrido benzina/GPL    IBM = ibrido benzina/metano
#   IGG = ibrido gasolio/GPL    IGM = ibrido gasolio/metano
# (IM e IGM non compaiono nei dati 2025: nessuna provincia ha valori
# per queste due colonne, quindi non sono presenti nel dataset).
COLONNE_ELETTRICO_IBRIDO = ["EL", "IB", "IG", "IBG", "IBM", "IGG"]

province_2025["elettrico_ibrido"] = province_2025[COLONNE_ELETTRICO_IBRIDO].sum(axis=1)
province_2025["quota_elettrico_ibrido"] = province_2025["elettrico_ibrido"] / province_2025["Totale"]

province_2025_sorted = province_2025.sort_values("elettrico_ibrido", ascending=False)
province_2025_sorted[["Provincia", "Totale", "elettrico_ibrido", "quota_elettrico_ibrido"]].head(10)

,Provincia,Totale,elettrico_ibrido,quota_elettrico_ibrido
66,ROMA,2965578,428934,0.144638
12,MILANO,1874010,284514,0.151821
0,TORINO,1649645,257939,0.156360
22,TRENTO,837647,245437,0.293008
50,FIRENZE,886723,237930,0.268325
42,BOLOGNA,646579,85732,0.132593
14,BRESCIA,860480,84043,0.097670
9,VARESE,616413,81184,0.131704
20,MONZA BRIANZA,595813,78087,0.131060
13,BERGAMO,734764,77084,0.104910


In [ ]:
# ============================================================
# CONFRONTO RAPIDO 2019 vs 2025
# ============================================================
# 2019: dal dataset ACI/MIT già caricato in auto_da_ricaricare.ipynb
# (tutte le categorie di veicolo, snapshot 31/12/2019): 409.397
# elettriche/ibride su 53.927.245 veicoli totali.
#
# Il confronto corretto per\u00f2 e' a parita' di categoria: il numero
# ufficiale ACI "Autoritratto 2019" per le sole AUTOVETTURE (non moto/
# altri mezzi) era 357.296 elettriche/ibride, fonte usata nella
# conversazione precedente per validare il dato 2019.

elettrico_ibrido_2019_autovetture = 357_296
elettrico_ibrido_2025_autovetture = int(province_2025["elettrico_ibrido"].sum())

crescita = elettrico_ibrido_2025_autovetture / elettrico_ibrido_2019_autovetture

print("Elettrico + ibrido, categoria AUTOVETTURE, Italia:")
print(f"  2019: {elettrico_ibrido_2019_autovetture:,}")
print(f"  2025: {elettrico_ibrido_2025_autovetture:,}")
print(f"  Crescita: {crescita:.1f}x in 6 anni")
print()
print("Nota: il totale 2025 qui sopra e' la somma per provincia calcolata")
print("da questo notebook; deve combaciare con il totale Italia gia'")
print("calcolato dalla dashboard ACI (riga 'Totale'):")
print(f"  EL+IB+IG+IBG+IBM+IGG dalla riga Totale: "
      f"{sum(to_int(totale_italia[c]) for c in COLONNE_ELETTRICO_IBRIDO):,}")

Elettrico + ibrido, categoria AUTOVETTURE, Italia:
  2019: 357,296
  2025: 4,015,707
  Crescita: 11.2x in 6 anni

Nota: il totale 2025 qui sopra e' la somma per provincia calcolata
da questo notebook; deve combaciare con il totale Italia gia'
calcolato dalla dashboard ACI (riga 'Totale'):
  EL+IB+IG+IBG+IBM+IGG dalla riga Totale: 4,015,707


In [ ]:
# ============================================================
# VEICOLI CHE NECESSITANO DI RICARICA — ELETTRICO + IBRIDO PLUG-IN
# ============================================================
# Per il progetto EV Charge Desert non tutte le "ibride" contano: le
# ibride tradizionali (full/mild hybrid) si ricaricano da sole in
# marcia e NON hanno bisogno di una colonnina, a differenza delle
# ibride plug-in, che si ricaricano alla presa esattamente come le
# elettriche pure. La colonna "ibrido" di ACI (IB, IG, IBG, IBM, IGG)
# non distingue plug-in da non plug-in: serve una stima.
#
# Dato UNRAE: il 27,6% delle auto ibride in circolazione sono plug-in.
# Applichiamo questa quota al totale ibrido di ogni provincia per
# stimare quante, di quelle ibride, necessitano davvero di ricarica.
# Fonte incerta a livello di singola provincia (il 27,6% e' una media
# nazionale UNRAE, non un dato per provincia: qui assumiamo che la
# quota plug-in/ibrido sia uniforme su tutto il territorio, un limite
# da tenere presente).
QUOTA_IBRIDO_PLUGIN_UNRAE = 0.276

COLONNE_IBRIDO = ["IB", "IG", "IBG", "IBM", "IGG"]  # tutte le ibride, non solo elettrico puro

domanda_ricarica_2025 = province_2025[["Provincia"]].copy()
domanda_ricarica_2025["elettrico"] = province_2025["EL"]
domanda_ricarica_2025["ibrido_totale"] = province_2025[COLONNE_IBRIDO].sum(axis=1)
domanda_ricarica_2025["ibrido_plugin_stimato"] = (
    domanda_ricarica_2025["ibrido_totale"] * QUOTA_IBRIDO_PLUGIN_UNRAE
).round().astype(int)
domanda_ricarica_2025["veicoli_da_ricaricare"] = (
    domanda_ricarica_2025["elettrico"] + domanda_ricarica_2025["ibrido_plugin_stimato"]
)

domanda_ricarica_2025 = domanda_ricarica_2025.sort_values(
    "veicoli_da_ricaricare", ascending=False
).reset_index(drop=True)

print(f"Totale Italia veicoli da ricaricare stimati (elettrico + ibrido plug-in): "
      f"{domanda_ricarica_2025['veicoli_da_ricaricare'].sum():,}")
domanda_ricarica_2025.head(10)

Totale Italia veicoli da ricaricare stimati (elettrico + ibrido plug-in): 1,370,592


,Provincia,elettrico,ibrido_totale,ibrido_plugin_stimato,veicoli_da_ricaricare
0,ROMA,41550,387384,106918,148468
1,MILANO,25911,258603,71374,97285
2,TRENTO,31918,213519,58931,90849
3,TORINO,17377,240562,66395,83772
4,FIRENZE,23250,214680,59252,82502
5,BRESCIA,10565,73478,20280,30845
6,BOLOGNA,7193,78539,21677,28870
7,VARESE,6919,74265,20497,27416
8,BERGAMO,8248,68836,18999,27247
9,BOLZANO,9620,61787,17053,26673


In [ ]:
# Salvataggio del file finale per provincia (elettrico + ibrido plug-in
# stimato), pronto per essere incrociato con colonnine/POI/sezioni.
OUTPUT_DOMANDA_RICARICA = "domanda_ricarica_2025_per_provincia.csv"

domanda_ricarica_2025.to_csv(OUTPUT_DOMANDA_RICARICA, index=False, encoding="utf-8")
print(f"Salvato: {OUTPUT_DOMANDA_RICARICA}")
print(f"{len(domanda_ricarica_2025)} righe (province)")

Salvato: domanda_ricarica_2025_per_provincia.csv
107 righe (province)
